# 4b — RAG Engine retrieval + grounded generation

**AWS-side equivalent:** Bedrock Knowledge Base **`retrieve`** only — Bedrock KB
owns both retrieve and generate behind one API, whereas with Vertex RAG Engine we
own the generation step (we call Gemini ourselves with the retrieval tool attached).

**GCP service:** Vertex AI RAG Engine (`rag.retrieval_query`) + Gemini grounded
generation via `Tool.from_retrieval`.

Depends on `03-rag-corpus.ipynb` — that notebook seeds
`config.gcp.rag_corpus_name`. If you skipped 03, this notebook will fail loudly
in the next cell.

> **API note (verified May 2026):** `vertexai.generative_models` deprecates on
> **2026-06-24** but `vertexai.rag` is GA and stays. The
> `Tool.from_retrieval(...)` glue lives in `generative_models` for the moment;
> we'll migrate to `google-genai`'s native RAG tool when it lands.


## Step 0: Imports + config (must have a corpus name)

In [ ]:
import json
from pathlib import Path

import vertexai
from vertexai import rag
from vertexai.generative_models import GenerativeModel, Tool

config_path = Path('config.json')
config = json.loads(config_path.read_text())
gcp = config.get('gcp', {})

PROJECT_ID  = gcp.get('project_id')
REGION      = gcp.get('region', 'us-central1')
MODEL_ID    = gcp.get('fast_model', 'gemini-2.5-flash')
CORPUS_NAME = gcp.get('rag_corpus_name')

assert PROJECT_ID, 'Run 01-setup-credentials.ipynb first.'
assert CORPUS_NAME, (
    'config.gcp.rag_corpus_name is not set. '
    'Run 03-rag-corpus.ipynb before this notebook — it creates and registers '
    'the corpus that 04b queries.'
)

vertexai.init(project=PROJECT_ID, location=REGION)

print(f'project : {PROJECT_ID}')
print(f'region  : {REGION}')
print(f'model   : {MODEL_ID}')
print(f'corpus  : {CORPUS_NAME}')

## Step 1: Raw retrieval (no generation)

`rag.retrieval_query` is the pure-retrieval surface — it returns chunks +
distance scores with no LLM call. This is the equivalent of Bedrock KB's
`retrieve` API. Use it when you want full control of the generation step
(custom prompt, custom model, custom citation rendering).

In [ ]:
retrieval_config = rag.RagRetrievalConfig(
    top_k=3,
    filter=rag.Filter(vector_distance_threshold=0.5),
)

response = rag.retrieval_query(
    rag_resources=[rag.RagResource(rag_corpus=CORPUS_NAME)],
    text='Who is on the Platform Engineering team?',
    rag_retrieval_config=retrieval_config,
)

contexts = list(response.contexts.contexts) if hasattr(response.contexts, 'contexts') else list(response.contexts)
print(f'Retrieved {len(contexts)} chunks:\n')
for i, ctx in enumerate(contexts, 1):
    dist = getattr(ctx, 'distance', None) or getattr(ctx, 'score', None)
    src = getattr(ctx, 'source_uri', '') or getattr(ctx, 'source_display_name', '')
    text = getattr(ctx, 'text', '')
    print(f'  {i}. distance={dist}  source={src}')
    print(f'     {text[:160].strip()}…')
    print()


## Step 2: Grounded generation via `Tool.from_retrieval`

Attach the corpus to a Gemini model as a retrieval tool. The model
decides when to query the corpus and inlines the chunks before
answering. Citations land on `response.candidates[0].grounding_metadata`.

In [ ]:
rag_tool = Tool.from_retrieval(
    retrieval=rag.Retrieval(
        source=rag.VertexRagStore(
            rag_resources=[rag.RagResource(rag_corpus=CORPUS_NAME)],
            rag_retrieval_config=retrieval_config,
        ),
    ),
)

model = GenerativeModel(MODEL_ID, tools=[rag_tool])

QUERY = 'Who is on the Platform Engineering team and what are their roles?'
answer = model.generate_content(QUERY)

print('--- ANSWER ---')
print(answer.text)
print()

cand = answer.candidates[0] if answer.candidates else None
gm = getattr(cand, 'grounding_metadata', None) if cand else None
if gm:
    print('--- CITATIONS ---')
    chunks = getattr(gm, 'grounding_chunks', []) or []
    for i, ch in enumerate(chunks, 1):
        retrieved = getattr(ch, 'retrieved_context', None)
        if retrieved:
            print(f'  [{i}] uri={getattr(retrieved, "uri", "")}  title={getattr(retrieved, "title", "")}')
else:
    print('(no grounding_metadata — model answered without consulting the corpus)')


## Step 3: Per-query knob exploration — top_k

`top_k=3` gives a focused answer with cleaner citations. `top_k=10`
gives the model more material — useful for "who *all* knows X" style
questions, at the cost of longer prompts (more tokens billed) and
sometimes noisier answers.

In [ ]:
def ask(query: str, top_k: int) -> str:
    cfg = rag.RagRetrievalConfig(
        top_k=top_k,
        filter=rag.Filter(vector_distance_threshold=0.5),
    )
    tool = Tool.from_retrieval(
        retrieval=rag.Retrieval(
            source=rag.VertexRagStore(
                rag_resources=[rag.RagResource(rag_corpus=CORPUS_NAME)],
                rag_retrieval_config=cfg,
            ),
        ),
    )
    m = GenerativeModel(MODEL_ID, tools=[tool])
    return m.generate_content(query).text

Q = 'List everyone in the org who works on platform or infrastructure.'

print('=== top_k=3 ===')
print(ask(Q, top_k=3))
print()
print('=== top_k=10 ===')
print(ask(Q, top_k=10))

## Step 4: Multi-corpus pattern (sketch)

In production you'd point `rag_resources` at multiple distinct corpora (e.g.
people + docs + decisions). **The SDK currently limits `VertexRagStore` to
1 `RagResource`**, so until that restriction lifts the workaround is to fan out
retrieval calls in the application layer and merge the chunks before generation.

The cell below uses a single corpus so it runs as-is.


In [ ]:
# SDK currently limits VertexRagStore to 1 RagResource — using single corpus here.
# Multi-corpus fan-out would be done at the application layer until the SDK lifts this.
multi_corpus_tool = Tool.from_retrieval(
    retrieval=rag.Retrieval(
        source=rag.VertexRagStore(
            rag_resources=[
                rag.RagResource(rag_corpus=CORPUS_NAME),
            ],
            rag_retrieval_config=rag.RagRetrievalConfig(
                top_k=5,
                filter=rag.Filter(vector_distance_threshold=0.5),
            ),
        ),
    ),
)

multi_model = GenerativeModel(MODEL_ID, tools=[multi_corpus_tool])
print(multi_model.generate_content(
    'Who manages the Platform Engineering team, and who reports to them?'
).text)


## Step 5: Cost & cleanup pointer

The corpus itself is cheap at idle — you pay for storage (small) and
per-query retrieval. Generation cost is whatever you'd pay for
`gemini-2.5-flash` calls regardless of the RAG attachment.

`19-cleanup` walks `tracked_resources` and uses the `rag_corpus` Python
deleter (`cleanup_utils._delete_rag_corpus`) to tear this down. The
staging-bucket files (uploaded in 03) are torn down via the
`gcs_bucket` gcloud deleter.

In [ ]:
print(f'Tracked corpus name : {CORPUS_NAME}')
print()
print('--- tracked_resources entry for this corpus ---')
tracked = gcp.get('tracked_resources', [])
match = [r for r in tracked if r.get('kind') == 'rag_corpus' and r.get('name') == CORPUS_NAME]
if match:
    print(json.dumps(match[0], indent=2))
else:
    print('(no rag_corpus entry found — did 03-rag-corpus.ipynb finish?)')

print()
print('Run 19-cleanup.ipynb to tear this corpus + staging bucket down.')